In [69]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
import clingo

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from api.data_utils import extract_from_json, aggregate_to_operator_day, generate_clingo_facts, generate_physical_instance

In [70]:
def generate_predictions_for_date(json_path, target_date, train_columns, trained_models, output_filename):
    df_raw = extract_from_json([json_path])
    df_agg = aggregate_to_operator_day(df_raw)
    
    df_today = df_agg[df_agg['planning_date'].astype(str).str.contains(target_date)].copy()
    
    if df_today.empty:
        print("Nessun dato trovato per questa data.")
        return
        
    today_ids = df_today['operator_id']
    X_today = df_today.drop(columns=['operator_id', 'planning_date', 'target_assignments'], errors='ignore')
    
    cat_cols = X_today.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    X_today = pd.get_dummies(X_today, columns=cat_cols, drop_first=True, dtype=int)
    
    X_today = X_today.reindex(columns=train_columns, fill_value=0)
    
    q10 = trained_models['q10'].predict(X_today)
    q50 = trained_models['q50'].predict(X_today)
    q90 = trained_models['q90'].predict(X_today)
    
    y_dummy = np.zeros(len(today_ids))
    
    generate_clingo_facts(
        X_test=df_today,
        y_test=pd.Series(y_dummy), 
        q10=q10,
        q50=q50,
        q90=q90,
        original_ids=today_ids, 
        output_filename=output_filename
    )

In [71]:
def run_neuro_symbolic_solver(rules_files_list, facts_file, ml_file, timeout_seconds=30.0):    
    ctl = clingo.Control(["--opt-strategy=usc,k,0,4", "--opt-usc-shrink=bin"])
    
    for r_file in rules_files_list:
        ctl.load(r_file)
        
    ctl.load(facts_file)
    if ml_file:
        ctl.load(ml_file)
    
    ctl.ground([("base", [])])
    
    best_cost = None
    assignments = []
    unassigned_count = 0
    
    def on_model(model):
        nonlocal best_cost, assignments, unassigned_count
        best_cost = model.cost
        assignments = [str(sym) for sym in model.symbols(shown=True)]
        unassigned_count = sum(1 for a in assignments if "assignment(-1" in a)
    
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout_seconds)
        if not finished:
            handle.cancel()
    
    return assignments, best_cost, unassigned_count

In [ ]:
def run_monthly_experiment(json_path, start_date, end_date, train_columns, trained_models, timeout=30.0):
    dates = pd.date_range(start=start_date, end=end_date)
    results = []
    
    os.makedirs("encodings/facts", exist_ok=True)
    
    for date_obj in dates:
        date_str = date_obj.strftime('%Y-%m-%d')
        print(f"GIORNO: {date_str} ", end="")
        
        try:
            generate_physical_instance(json_path, date_str, "encodings/facts/day_facts.lp")
        except Exception as e:
            print(f"[Saltato: Errore estrazione]")
            continue
            
        with open("encodings/facts/day_facts.lp", 'r') as f:
            if len(f.readlines()) < 10:
                print(f"[Saltato: Ospedale Vuoto]")
                continue
        
        try:
            generate_predictions_for_date(json_path, date_str, train_columns, trained_models, "encodings/facts/ml_facts.lp")
        except Exception as e:
            print(f"[Saltato: Errore ML]")
            continue
            
        ass_base, cost_base, unass_base = run_neuro_symbolic_solver(
            rules_files_list=["encodings/base_rules.lp"],
            facts_file="encodings/facts/day_facts.lp",
            ml_file=None,
            timeout_seconds=timeout
        )
    
        ass_ns, cost_ns, unass_ns = run_neuro_symbolic_solver(
            rules_files_list=["encodings/base_rules.lp", "encodings/ml_rules.lp"],
            facts_file="encodings/facts/day_facts.lp",
            ml_file="encodings/facts/ml_facts.lp",
            timeout_seconds=timeout
        )
        
        cost_base_list = cost_base if cost_base is not None else [0, 0, 0]
        cost_ns_list = cost_ns if cost_ns is not None else [0, 0, 0]
        
        results.append({
            'Data': date_str,
            'Pazienti_A_Terra_Baseline': unass_base,
            'Pazienti_A_Terra_NS': unass_ns,
            'Assegnazioni_Totali_Baseline': len(ass_base) - unass_base,
            'Assegnazioni_Totali_NS': len(ass_ns) - unass_ns,
            'Costo_W2_Baseline': cost_base_list[1] if len(cost_base_list) > 1 else 0,
            'Costo_W2_NS': cost_ns_list[1] if len(cost_ns_list) > 1 else 0
        })
        print(f"[OK]")

    return pd.DataFrame(results)

In [73]:
try:
    trained_models = joblib.load("saved_models/best_quantiles_model.pkl")
    train_columns = joblib.load("saved_models/train_columns.pkl")
except FileNotFoundError:
    print("File dei modelli non trovati.")

data_url = 'data/anon_boardHistory_Org6_2023-01-01_2023-01-31.json'
df_esperimento = run_monthly_experiment(
    json_path=data_url, 
    start_date='2023-01-01', 
    end_date='2023-01-31', 
    train_columns=train_columns, 
    trained_models=trained_models,
    timeout=20.0
)

display(df_esperimento)


GIORNO: 2023-01-01 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-02 [OK]
GIORNO: 2023-01-03 [OK]
GIORNO: 2023-01-04 [OK]
GIORNO: 2023-01-05 [OK]
GIORNO: 2023-01-06 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-07 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-08 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-09 [OK]
GIORNO: 2023-01-10 [OK]
GIORNO: 2023-01-11 [OK]
GIORNO: 2023-01-12 [OK]
GIORNO: 2023-01-13 [OK]
GIORNO: 2023-01-14 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-15 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-16 [OK]
GIORNO: 2023-01-17 [OK]
GIORNO: 2023-01-18 [OK]
GIORNO: 2023-01-19 [OK]
GIORNO: 2023-01-20 [OK]
GIORNO: 2023-01-21 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-22 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-23 [OK]
GIORNO: 2023-01-24 [OK]
GIORNO: 2023-01-25 [OK]
GIORNO: 2023-01-26 [OK]
GIORNO: 2023-01-27 [OK]
GIORNO: 2023-01-28 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-29 [Saltato: Ospedale Vuoto]
GIORNO: 2023-01-30 [OK]
GIORNO: 2023-01-31 [OK]


,Data,Pazienti_A_Terra_Baseline,Pazienti_A_Terra_NS,Assegnazioni_Totali_Baseline,Assegnazioni_Totali_NS,Costo_W2_Baseline,Costo_W2_NS
0,2023-01-02,1,1,88,88,1,1
1,2023-01-03,1,1,60,60,1,1
2,2023-01-04,0,0,96,96,0,0
3,2023-01-05,0,0,99,99,0,0
4,2023-01-09,0,0,100,100,0,0
5,2023-01-10,0,0,102,102,0,0
6,2023-01-11,0,0,105,105,0,0
7,2023-01-12,0,0,112,112,0,98
8,2023-01-13,0,0,114,114,0,0
9,2023-01-16,0,0,112,112,0,0


In [ ]:
import requests
import json

def test_api_optimize():
    with open("../data/anon_boardHistory_Org6_2023-01-01_2023-01-31.json", 'r', encoding='utf-8') as f:
        hospital_payload = json.load(f)

    if isinstance(hospital_payload, dict):
        hospital_payload = [hospital_payload]

    richiesta = {
        "target_date": "2023-01-09",
        "use_ml": True,
        "payload": hospital_payload
    }

    print("Invio richiesta all'API in corso...")
    risposta = requests.post("http://127.0.0.1:8000/optimize", json=richiesta)

    if risposta.status_code == 200:
        dati = risposta.json()
        print("✅ API RISPONDE PERFETTAMENTE!")
        print(f"Stato: {dati['status']}")
        print(f"Costo: {dati['cost']}")
        print(f"Pazienti non assegnati: {dati['unassigned_patients']}")
    else:
        print(f"❌ Errore API: {risposta.status_code}")
        print(risposta.text)

# test_api_optimize()

Invio richiesta all'API in corso...
✅ API RISPONDE PERFETTAMENTE!
Stato: SAT
Costo: [120, 0, 14]
Pazienti non assegnati: 0
